# 02 — EDA and SQL Analysis

## Purpose

This notebook performs the first complete exploratory analysis of the Olist e-commerce database.

The analysis is deliberately split into two perspectives:

1. **SQL-first business analysis** — use PostgreSQL/Supabase for relational aggregations and joins.
2. **Python EDA** — retrieve analytical result sets and visualize them with pandas and Plotly.

This keeps the project aligned with a real analytics workflow instead of loading every raw table into one giant pandas DataFrame.

> This notebook is descriptive analytics only. Machine-learning feature engineering happens later.

## Questions this notebook answers

### Customers
- How many customers and orders are present?
- What proportion of customers are one-time vs repeat buyers?
- How is customer spending distributed?
- Which states generate the most customer revenue?

### Products
- Which categories generate the most revenue?
- Which products are most popular?
- What does the price distribution look like?

### Operations
- How long does delivery take?
- Which categories have slower delivery?
- How does actual delivery compare with estimated delivery?

### Reviews
- What is the review-score distribution?
- Which categories have stronger or weaker customer satisfaction?

### Payments
- Which payment methods are most common?
- What is the distribution of payment values?

### Time
- How does revenue change over time?
- How does active-customer activity change over time?

In [ ]:
from pathlib import Path
import sys
import warnings

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import text

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.database import get_engine

engine = get_engine()

print(f"Project root: {PROJECT_ROOT}")
print("Database engine created successfully.")

## 1. Database overview

In [ ]:
overview_query = text("""
SELECT
    'customers' AS entity,
    COUNT(*) AS row_count
FROM olist_customers

UNION ALL
SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL
SELECT 'payments', COUNT(*) FROM olist_order_payments
UNION ALL
SELECT 'reviews', COUNT(*) FROM olist_order_reviews
UNION ALL
SELECT 'products', COUNT(*) FROM olist_products
UNION ALL
SELECT 'sellers', COUNT(*) FROM olist_sellers
UNION ALL
SELECT 'geolocation', COUNT(*) FROM olist_geolocation
UNION ALL
SELECT 'category_translation', COUNT(*)
FROM product_category_name_translation
ORDER BY entity
""")

with engine.connect() as connection:
    overview = pd.read_sql(overview_query, connection)

display(overview)

## 2. Order status analysis

In [ ]:
query = text("""
SELECT
    order_status,
    COUNT(*) AS order_count,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM olist_orders
GROUP BY order_status
ORDER BY order_count DESC
""")

with engine.connect() as connection:
    order_status = pd.read_sql(query, connection)

display(order_status)

fig = px.bar(
    order_status,
    x="order_status",
    y="order_count",
    title="Order Status Distribution",
    labels={
        "order_status": "Order Status",
        "order_count": "Number of Orders",
    },
)
fig.show()

## 3. One-time vs repeat customers

In [ ]:
query = text("""
WITH customer_orders AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS order_count
    FROM olist_customers c
    JOIN olist_orders o
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
)
SELECT
    CASE
        WHEN order_count = 1 THEN 'One-time customer'
        ELSE 'Repeat customer'
    END AS customer_type,
    COUNT(*) AS customer_count,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM customer_orders
GROUP BY customer_type
ORDER BY customer_count DESC
""")

with engine.connect() as connection:
    customer_type = pd.read_sql(query, connection)

display(customer_type)

fig = px.pie(
    customer_type,
    names="customer_type",
    values="customer_count",
    title="One-time vs Repeat Customers",
    hole=0.45,
)
fig.show()

## 4. Customer spending distribution

In [ ]:
query = text("""
SELECT
    c.customer_unique_id,
    SUM(oi.price + oi.freight_value) AS total_spend
FROM olist_customers c
JOIN olist_orders o
    ON c.customer_id = o.customer_id
JOIN olist_order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_unique_id
""")

with engine.connect() as connection:
    customer_spend = pd.read_sql(query, connection)

customer_spend["total_spend"] = pd.to_numeric(
    customer_spend["total_spend"],
    errors="coerce"
)

display(customer_spend.describe().T)

fig = px.histogram(
    customer_spend,
    x="total_spend",
    nbins=60,
    title="Customer Spending Distribution",
    labels={"total_spend": "Customer Spend"},
)
fig.show()

### Log-scale view

E-commerce customer spend is usually highly skewed. A log-scale visualization helps inspect the long tail without deleting high-value customers.

In [ ]:
customer_spend_plot = customer_spend.copy()
customer_spend_plot["log_spend"] = np.log1p(customer_spend_plot["total_spend"])

fig = px.histogram(
    customer_spend_plot,
    x="log_spend",
    nbins=60,
    title="Log-Transformed Customer Spending Distribution",
    labels={"log_spend": "log(1 + customer spend)"},
)
fig.show()

## 5. Top customer states by revenue

In [ ]:
query = text("""
SELECT
    c.customer_state,
    COUNT(DISTINCT c.customer_unique_id) AS customer_count,
    COUNT(DISTINCT o.order_id) AS order_count,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS revenue
FROM olist_customers c
JOIN olist_orders o
    ON c.customer_id = o.customer_id
JOIN olist_order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_state
ORDER BY revenue DESC
""")

with engine.connect() as connection:
    state_revenue = pd.read_sql(query, connection)

display(state_revenue)

fig = px.bar(
    state_revenue.head(15),
    x="customer_state",
    y="revenue",
    title="Top Customer States by Revenue",
    labels={
        "customer_state": "State",
        "revenue": "Revenue",
    },
)
fig.show()

## 6. Monthly revenue and active customers

In [ ]:
query = text("""
SELECT
    DATE_TRUNC(
        'month',
        o.order_purchase_timestamp
    ) AS purchase_month,
    COUNT(DISTINCT c.customer_unique_id) AS active_customers,
    COUNT(DISTINCT o.order_id) AS orders,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS revenue
FROM olist_customers c
JOIN olist_orders o
    ON c.customer_id = o.customer_id
JOIN olist_order_items oi
    ON o.order_id = oi.order_id
GROUP BY purchase_month
ORDER BY purchase_month
""")

with engine.connect() as connection:
    monthly = pd.read_sql(query, connection)

monthly["purchase_month"] = pd.to_datetime(monthly["purchase_month"])

display(monthly)

fig = px.line(
    monthly,
    x="purchase_month",
    y="revenue",
    markers=True,
    title="Monthly Revenue",
    labels={
        "purchase_month": "Month",
        "revenue": "Revenue",
    },
)
fig.show()

fig = px.line(
    monthly,
    x="purchase_month",
    y="active_customers",
    markers=True,
    title="Monthly Active Customers",
    labels={
        "purchase_month": "Month",
        "active_customers": "Active Customers",
    },
)
fig.show()

## 7. Category revenue analysis

In [ ]:
query = text("""
SELECT
    COALESCE(
        t.product_category_name_english,
        p.product_category_name,
        'Unknown'
    ) AS category,
    COUNT(DISTINCT p.product_id) AS product_count,
    COUNT(DISTINCT oi.order_id) AS order_count,
    COUNT(*) AS item_count,
    ROUND(SUM(oi.price), 2) AS revenue,
    ROUND(AVG(oi.price), 2) AS average_price
FROM olist_products p
JOIN olist_order_items oi
    ON p.product_id = oi.product_id
LEFT JOIN product_category_name_translation t
    ON p.product_category_name = t.product_category_name
GROUP BY category
ORDER BY revenue DESC
""")

with engine.connect() as connection:
    category_revenue = pd.read_sql(query, connection)

display(category_revenue.head(20))

fig = px.bar(
    category_revenue.head(20).sort_values("revenue"),
    x="revenue",
    y="category",
    orientation="h",
    title="Top Product Categories by Revenue",
    labels={
        "revenue": "Revenue",
        "category": "Category",
    },
)
fig.show()

## 8. Product price distribution

In [ ]:
query = text("""
SELECT
    price,
    freight_value
FROM olist_order_items
WHERE price IS NOT NULL
""")

with engine.connect() as connection:
    price_data = pd.read_sql(query, connection)

price_data["price"] = pd.to_numeric(price_data["price"], errors="coerce")
price_data["freight_value"] = pd.to_numeric(
    price_data["freight_value"],
    errors="coerce"
)

display(price_data[["price", "freight_value"]].describe().T)

fig = px.histogram(
    price_data,
    x="price",
    nbins=80,
    title="Order Item Price Distribution",
    labels={"price": "Item Price"},
)
fig.show()

## 9. Review-score analysis

In [ ]:
query = text("""
SELECT
    review_score,
    COUNT(*) AS review_count,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM olist_order_reviews
GROUP BY review_score
ORDER BY review_score
""")

with engine.connect() as connection:
    reviews = pd.read_sql(query, connection)

display(reviews)

fig = px.bar(
    reviews,
    x="review_score",
    y="review_count",
    title="Review Score Distribution",
    labels={
        "review_score": "Review Score",
        "review_count": "Reviews",
    },
)
fig.show()

## 10. Category review performance

In [ ]:
query = text("""
SELECT
    COALESCE(
        t.product_category_name_english,
        p.product_category_name,
        'Unknown'
    ) AS category,
    COUNT(DISTINCT r.review_id) AS review_count,
    ROUND(AVG(r.review_score), 2) AS average_review_score
FROM olist_products p
JOIN olist_order_items oi
    ON p.product_id = oi.product_id
JOIN olist_order_reviews r
    ON oi.order_id = r.order_id
LEFT JOIN product_category_name_translation t
    ON p.product_category_name = t.product_category_name
GROUP BY category
HAVING COUNT(DISTINCT r.review_id) >= 20
ORDER BY average_review_score
""")

with engine.connect() as connection:
    category_reviews = pd.read_sql(query, connection)

display(category_reviews.head(20))

fig = px.bar(
    category_reviews.head(15).sort_values("average_review_score"),
    x="average_review_score",
    y="category",
    orientation="h",
    title="Categories with Lower Average Review Scores",
    labels={
        "average_review_score": "Average Review Score",
        "category": "Category",
    },
)
fig.show()

## 11. Payment-method analysis

In [ ]:
query = text("""
SELECT
    payment_type,
    COUNT(*) AS payment_records,
    COUNT(DISTINCT order_id) AS orders,
    ROUND(SUM(payment_value), 2) AS payment_value,
    ROUND(AVG(payment_value), 2) AS average_payment
FROM olist_order_payments
GROUP BY payment_type
ORDER BY payment_value DESC
""")

with engine.connect() as connection:
    payments = pd.read_sql(query, connection)

display(payments)

fig = px.bar(
    payments,
    x="payment_type",
    y="payment_value",
    title="Payment Value by Payment Method",
    labels={
        "payment_type": "Payment Method",
        "payment_value": "Payment Value",
    },
)
fig.show()

## 12. Delivery-time analysis

In [ ]:
query = text("""
SELECT
    o.order_id,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    EXTRACT(
        EPOCH FROM (
            o.order_delivered_customer_date
            - o.order_purchase_timestamp
        )
    ) / 86400.0 AS delivery_days,
    EXTRACT(
        EPOCH FROM (
            o.order_delivered_customer_date
            - o.order_estimated_delivery_date
        )
    ) / 86400.0 AS delivery_vs_estimate_days
FROM olist_orders o
WHERE o.order_delivered_customer_date IS NOT NULL
""")

with engine.connect() as connection:
    delivery = pd.read_sql(query, connection)

delivery["delivery_days"] = pd.to_numeric(
    delivery["delivery_days"],
    errors="coerce"
)
delivery["delivery_vs_estimate_days"] = pd.to_numeric(
    delivery["delivery_vs_estimate_days"],
    errors="coerce"
)

display(
    delivery[
        ["delivery_days", "delivery_vs_estimate_days"]
    ].describe().T
)

fig = px.histogram(
    delivery,
    x="delivery_days",
    nbins=70,
    title="Delivery Time Distribution",
    labels={"delivery_days": "Delivery Days"},
)
fig.show()

## 13. Delivery performance

Interpretation:

- `delivery_vs_estimate_days < 0` means delivered before the estimated date.
- `delivery_vs_estimate_days > 0` means delivered after the estimated date.
- `0` means delivered on the estimated date.

In [ ]:
delivery_summary = pd.DataFrame({
    "metric": [
        "Average delivery days",
        "Median delivery days",
        "Average days vs estimate",
        "% delivered before estimate",
        "% delivered after estimate",
    ],
    "value": [
        delivery["delivery_days"].mean(),
        delivery["delivery_days"].median(),
        delivery["delivery_vs_estimate_days"].mean(),
        (delivery["delivery_vs_estimate_days"] < 0).mean() * 100,
        (delivery["delivery_vs_estimate_days"] > 0).mean() * 100,
    ],
})

delivery_summary["value"] = delivery_summary["value"].round(2)
display(delivery_summary)

## 14. Seller revenue concentration

In [ ]:
query = text("""
SELECT
    s.seller_id,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS order_count,
    COUNT(DISTINCT oi.product_id) AS product_count,
    ROUND(SUM(oi.price), 2) AS revenue
FROM olist_sellers s
JOIN olist_order_items oi
    ON s.seller_id = oi.seller_id
GROUP BY s.seller_id, s.seller_state
ORDER BY revenue DESC
""")

with engine.connect() as connection:
    seller_revenue = pd.read_sql(query, connection)

display(seller_revenue.head(20))

seller_revenue["revenue"] = pd.to_numeric(
    seller_revenue["revenue"],
    errors="coerce"
)

fig = px.histogram(
    seller_revenue,
    x="revenue",
    nbins=60,
    title="Seller Revenue Distribution",
    labels={"revenue": "Seller Revenue"},
)
fig.show()

## 15. Customer revenue concentration

In [ ]:
customer_revenue_query = text("""
WITH customer_revenue AS (
    SELECT
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value) AS total_spend
    FROM olist_customers c
    JOIN olist_orders o
        ON c.customer_id = o.customer_id
    JOIN olist_order_items oi
        ON o.order_id = oi.order_id
    GROUP BY c.customer_unique_id
),
ranked AS (
    SELECT
        *,
        SUM(total_spend) OVER (
            ORDER BY total_spend DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_revenue,
        SUM(total_spend) OVER () AS total_revenue
    FROM customer_revenue
)
SELECT
    customer_unique_id,
    total_spend,
    100.0 * cumulative_revenue / total_revenue
        AS cumulative_revenue_percentage
FROM ranked
ORDER BY total_spend DESC
""")

with engine.connect() as connection:
    concentration = pd.read_sql(customer_revenue_query, connection)

concentration["total_spend"] = pd.to_numeric(
    concentration["total_spend"],
    errors="coerce"
)
concentration["cumulative_revenue_percentage"] = pd.to_numeric(
    concentration["cumulative_revenue_percentage"],
    errors="coerce"
)

display(concentration.head(20))

fig = px.line(
    concentration.reset_index(),
    x="index",
    y="cumulative_revenue_percentage",
    title="Cumulative Customer Revenue Concentration",
    labels={
        "index": "Customers Ranked by Spend",
        "cumulative_revenue_percentage": "Cumulative Revenue %",
    },
)
fig.show()

## 16. SQL insight summary

In [ ]:
# Generate a compact numeric summary for later interpretation.

summary = {
    "total_customers": int(overview.loc[
        overview["entity"] == "customers", "row_count"
    ].iloc[0]),
    "total_orders": int(overview.loc[
        overview["entity"] == "orders", "row_count"
    ].iloc[0]),
    "total_products": int(overview.loc[
        overview["entity"] == "products", "row_count"
    ].iloc[0]),
    "total_sellers": int(overview.loc[
        overview["entity"] == "sellers", "row_count"
    ].iloc[0]),
    "total_reviews": int(overview.loc[
        overview["entity"] == "reviews", "row_count"
    ].iloc[0]),
    "total_order_items": int(overview.loc[
        overview["entity"] == "order_items", "row_count"
    ].iloc[0]),
    "total_revenue": float(
        customer_spend["total_spend"].sum()
    ),
    "median_customer_spend": float(
        customer_spend["total_spend"].median()
    ),
    "average_delivery_days": float(
        delivery["delivery_days"].mean()
    ),
    "median_delivery_days": float(
        delivery["delivery_days"].median()
    ),
}

summary_df = pd.DataFrame(
    [{"metric": key, "value": value} for key, value in summary.items()]
)

display(summary_df)

# EDA Conclusions

Use the outputs above to document business findings based on the actual database results.

The important discipline is:

- Do not invent business conclusions before observing the data.
- Do not remove outliers merely because they look unusual.
- Distinguish data-quality problems from legitimate long-tail behavior.
- Keep SQL aggregation in PostgreSQL where possible.
- Use Python primarily for statistical inspection and visualization.

The next stage will transform these findings into analytical features and modeling datasets.